In [ ]:
# 1 — cfg
import os, sys, json, shutil
from pathlib import Path

DRIVE_FOLDER = "ProtoFormer_out"
REPO = "/content/ProtoFormer"
GIT_URL = "https://github.com/pratamabintang/PF.git"
BRANCH = "main"
DATA = Path("/content/data")
SAM2_CKPT = "/content/sam2_hiera_l.pth"
SAM2_DRIVE_SRC = "/content/drive/MyDrive/project/checkpoints/sam2_hiera_large.pt"

# hyperparameter — ubah di sini saja
HP = dict(
    benchmark="pascal", fold=0,
    bsz=16, lr=1e-3, niter=3000, nworker=2, reduce_dim=64,
    amp="auto", patience=20, val_interval=3,
)

print(HP)


In [ ]:
# 2 — drive + repo + deps
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
OUT = Path(f"/content/drive/MyDrive/{DRIVE_FOLDER}")
OUT.mkdir(parents=True, exist_ok=True)
try:
    import git
except ImportError:
    import subprocess; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "GitPython"], check=True)
    import git
print("OUT:", OUT)


In [ ]:
# 3 — repo clone/pull
if not Path(REPO).exists():
    git.Repo.clone_from(GIT_URL, REPO, branch=BRANCH)
else:
    repo = git.Repo(REPO)
    repo.git.fetch("origin")
    repo.git.checkout(BRANCH)
    repo.git.reset("--hard", f"origin/{BRANCH}")
    repo.git.clean("-fd")
os.chdir(REPO)
assert Path(REPO, "train.py").exists()
print("repo OK", REPO, "sam2 ada:", Path(REPO, "model/sam2").exists())


In [ ]:
# 4 — deps
import subprocess
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
                   capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-3000:]
import torch, torchvision
print(torch.__version__, torchvision.__version__)


In [ ]:
# 5 — VOC2012 dari HuggingFace (tanpa subprocess)
from datasets import load_dataset
voc = DATA / "VOC2012"
for d in (voc / "JPEGImages", voc / "SegmentationClassAug", voc / "SegmentationClass"):
    d.mkdir(parents=True, exist_ok=True)
jpg_dir, seg_dir, img_dir = (voc / "JPEGImages", voc / "SegmentationClassAug", voc / "SegmentationClass")
if len(list(jpg_dir.glob("*.jpg"))) < 1000 or len(list(seg_dir.glob("*.png"))) < 1000:
    ds = load_dataset("shijli/voc2012", "segmentation_aug")
    seen = set()
    for split_name in ("train", "validation"):
        sp = ds[split_name]
        for i in range(len(sp)):
            row = sp[i]
            img_id = row["id"]
            if img_id in seen:
                continue
            seen.add(img_id)
            row["image"].convert("RGB").save(str(jpg_dir / f"{img_id}.jpg"), "JPEG", quality=95)
            row["mask"].save(str(seg_dir / f"{img_id}.png"), "PNG")
    ds2 = load_dataset("shijli/voc2012", "segmentation")
    for split_name in ("train", "validation"):
        sp = ds2[split_name]
        for i in range(len(sp)):
            row = sp[i]
            row["mask"].save(str(img_dir / f"{row['id']}.png"), "PNG")
    del ds, ds2
n_jpg = len(list(jpg_dir.glob("*.jpg")))
n_aug = len(list(seg_dir.glob("*.png")))
n_off = len(list(img_dir.glob("*.png")))
assert n_jpg >= 11831, f"JPEGImages kurang: {n_jpg}"
assert n_aug >= 11831, f"SegmentationClassAug kurang: {n_aug}"
assert n_off >= 2813, f"SegmentationClass kurang: {n_off}"
print("VOC OK", n_jpg, n_aug, n_off)


In [ ]:
# 6 — SAM2 checkpoint dari Drive
if Path(SAM2_DRIVE_SRC).exists():
    shutil.copy(SAM2_DRIVE_SRC, SAM2_CKPT)
assert Path(SAM2_CKPT).exists() and Path(SAM2_CKPT).stat().st_size > 100 * 1024 * 1024, \
    f"SAM2 rusak: {Path(SAM2_CKPT).stat().st_size if Path(SAM2_CKPT).exists() else 'missing'} byte"
print("SAM2 OK", Path(SAM2_CKPT).stat().st_size // 1024 // 1024, "MB")


In [ ]:
# 7 — train 1-shot (CLI, foreground — kernel block sampai selesai)
# PYTHONPATH=model: sam2 vendored di model/sam2, bukan pip package
!cd /content/ProtoFormer && PYTHONPATH=model python train.py --datapath /content/data --benchmark pascal --fold 0 --bsz 16 --lr 0.001 --niter 3000 --nworker 2 --reduce_dim 64 --img_size 512 --amp auto --sam2_checkpoint /content/sam2_hiera_l.pth --patience 20 --val_interval 3 --shot 1 --logpath pascal_fold0_1shot


In [ ]:
# 8 — train 5-shot (CLI, foreground — kernel block sampai selesai)
# PYTHONPATH=model: sam2 vendored di model/sam2, bukan pip package
!cd /content/ProtoFormer && PYTHONPATH=model python train.py --datapath /content/data --benchmark pascal --fold 0 --bsz 16 --lr 0.001 --niter 3000 --nworker 2 --reduce_dim 64 --img_size 512 --amp auto --sam2_checkpoint /content/sam2_hiera_l.pth --patience 20 --val_interval 3 --shot 5 --logpath pascal_fold0_5shot


In [ ]:
# 9 — save: salin weight + histori ke Drive
import zipfile
best = sorted(Path(REPO).glob("logs/*/best_model.pt"))  # Logger.save_model_miou -> logs/<logpath>/best_model.pt
print("weight:", [str(b) for b in best])
files = []
for b in best:
    dst = OUT / b.name
    shutil.copy(b, dst)
    files.append(dst)
for p in sorted(Path(REPO).rglob("*.json")):
    if "hist" in p.name or "config" in p.name:
        shutil.copy(p, OUT / p.name); files.append(OUT / p.name)
for f in files:
    print(f.name, f.stat().st_size // 1024 // 1024, "MB")
assert files, "tidak ada weight tersimpan"
with zipfile.ZipFile(OUT / "protoformer_out.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, f.name)
print("saved:", OUT / "protoformer_out.zip")
